In [1]:
import torch
from torch.utils.data import DataLoader, Dataset
import pandas as pd 
import numpy as np
from PIL import Image
from torchvision import transforms

In [2]:
label_encoding = {"healthy": 0, "scab": 1, "rust": 2, "frog_eye_leaf_spot": 3, "powdery_mildew": 4}

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])


In [3]:
class PlantPathologyDataset(Dataset):
    def __init__(self, subset, transform=None):
        data = pd.read_csv(f"dataset/{subset}.csv")
        self.filepaths = data["image"].values
        self.target = data["labels"].values
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        
        img_path = f"dataset/images/{self.filepaths[idx]}"
        image = Image.open(img_path)
        
        label = self.target[idx]
        label = label_encoding[label]
        label = torch.tensor(label, dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label

In [4]:
val_dataset = PlantPathologyDataset("val", transform=transform)

In [5]:
loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [6]:
for images, labels in loader:
    print(images.shape, labels.shape)
    break

torch.Size([32, 3, 224, 224]) torch.Size([32])
